### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [6]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots don’t “talk” in the same way humans use language; they **mimic** sounds they hear, including human speech. Their ability to do this comes from a combination of anatomy, brain wiring, and social behavior:\n\n| Factor | How it helps parrots mimic speech |\n|--------|-----------------------------------|\n| **Vocal anatomy – the syrinx** | The syrinx (the bird’s voice box) sits at the base of the trachea and is far more flexible than a mammalian larynx. Parrots can control the tension of the membranes and the airflow with exquisite precision, allowing them to produce a huge range of pitches, tones, and rapid modulations. |\n| **Brain structures – vocal learning centers** | Parrots belong to the order *Psittaciformes*, which, like songbirds and some mammals (e.g., humans, cetaceans), have a well‑developed “song system” in the forebrain (especially the **nidopallium** and **arcopallium**). These regions enable **vocal learning**—the ability to hear a sound, store a

In [7]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [8]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks: "What\'s the weather like in Boston?" We need to get weather. Use function get_weather.', 'tool_calls': [{'id': 'fc_44024e5c-5e8f-4c00-81c7-44d7c7c663dd', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 127, 'total_tokens': 178, 'completion_time': 0.106630017, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.004833714, 'prompt_tokens_details': None, 'queue_time': 0.074243307, 'total_time': 0.111463731}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_19b184c447', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a18032-d3df-75e2-8177-664176de0b97-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_44024e5c-5e8f-4c00-81c7-44d7c7c663dd', 'type': 'tool_call'}] invali

### Tool Execution Loops

In [9]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The current weather in Boston is sunny. Enjoy your day!


In [10]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in Boston. Use function get_weather.', 'tool_calls': [{'id': 'fc_383eea8a-d0df-476c-b5d2-5bdb996f63a3', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 126, 'total_tokens': 166, 'completion_time': 0.083599361, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.026137698, 'prompt_tokens_details': None, 'queue_time': 0.138612478, 'total_time': 0.109737059}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f640395b96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a18032-d4e5-7109-a9fd-74c905f7d541-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_383eea8a-d0df-476c-b5d2-5bdb996f6